# Apply self-heating correction to open-path fluxes (NEE, 2004-2019)

# Imports

In [4]:
from datetime import datetime
from pathlib import Path
import pandas as pd
from diive.core.io.files import save_parquet, load_parquet
from diive.pkgs.flux.selfheating import ScopPhysics, ScopOptimizer, ScopApplicator

# Docstring

In [2]:
help(ScopApplicator)

Help on class ScopApplicator in module diive.pkgs.flux.selfheating:

class ScopApplicator(builtins.object)
 |  ScopApplicator(flux_type: Literal['CO2', 'H2O'], fct_unsc: pandas.core.series.Series, scaling_factors_df: pandas.core.frame.DataFrame, flux_openpath: pandas.core.series.Series, flux_closedpath: pandas.core.series.Series, classvar: pandas.core.series.Series, daytime: pandas.core.series.Series, swin: pandas.core.series.Series, ts: pandas.core.series.Series, ra: pandas.core.series.Series, rho_d: pandas.core.series.Series, latent_heat_vaporization: Optional[pandas.core.series.Series] = None)
 |  
 |  Methods defined here:
 |  
 |  __init__(self, flux_type: Literal['CO2', 'H2O'], fct_unsc: pandas.core.series.Series, scaling_factors_df: pandas.core.frame.DataFrame, flux_openpath: pandas.core.series.Series, flux_closedpath: pandas.core.series.Series, classvar: pandas.core.series.Series, daytime: pandas.core.series.Series, swin: pandas.core.series.Series, ts: pandas.core.series.Series

# Load data

In [7]:
SOURCEDIR = r"../../20_MERGE_DATA"
FILENAME = r"21.4_FLUXES_L1_noSHC_IRGA75+METEO7.parquet"
FILEPATH = Path(SOURCEDIR) / FILENAME
print(f"Data will be loaded from the following file:\n{FILEPATH}")
df = load_parquet(filepath=FILEPATH)
df

Data will be loaded from the following file:
..\..\20_MERGE_DATA\21.4_FLUXES_L1_noSHC_IRGA75+METEO7.parquet
Loaded .parquet file ..\..\20_MERGE_DATA\21.4_FLUXES_L1_noSHC_IRGA75+METEO7.parquet (0.386 seconds).
    --> Detected time resolution of <30 * Minutes> / 30min 


,AIR_CP,AIR_DENSITY,AIR_MV,AIR_RHO_CP,AOA_METHOD,AXES_ROTATION_METHOD,BADM_HEIGHTC,...,W_T_SONIC_COV_IBROM_N1626,W_UNROT,W_U_COV,W_VM97_TEST,W_ZCD,ZL,ZL_UNCORR
TIMESTAMP_MIDDLE,,,,,,,,,,,,,,,
2005-01-01 00:15:00,1008.03,1.19492,0.024191,1204.51,0.0,1.0,37.0,...,-0.019764,0.045938,-0.105746,800000000.0,11.0,0.118888,0.137374
2005-01-01 00:45:00,1008.11,1.19419,0.024204,1203.88,0.0,1.0,37.0,...,0.022155,0.403188,-0.423825,800000000.0,5.0,-0.044524,-0.048667
2005-01-01 01:15:00,1008.17,1.19318,0.024223,1202.93,0.0,1.0,37.0,...,0.009001,0.189884,-0.266300,800000000.0,7.0,-0.011929,-0.012089
2005-01-01 01:45:00,1008.24,1.19202,0.024246,1201.84,0.0,1.0,37.0,...,0.016574,0.227756,-0.267261,800000000.0,9.0,-0.039767,-0.038691
2005-01-01 02:15:00,1008.29,1.19110,0.024264,1200.97,0.0,1.0,37.0,...,0.030212,0.191151,-0.193863,800000000.0,13.0,-0.093878,-0.095240
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2019-12-31 21:45:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2019-12-31 22:15:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2019-12-31 22:45:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN


# Calculate unscaled flux correction term with `ScopPhysics`

In [8]:
physics = ScopPhysics(
    flux_type="CO2",
    ta=df["TA_T1_47_1"].copy(),
    gas_density=df["CO2_MOLAR_DENSITY"].copy() * 1000,  # Requires umol m-3
    rho_a=df["AIR_DENSITY"].copy(),
    rho_v=df["VAPOR_DENSITY"].copy(),
    u=df["U"].copy(),
    c_p=df["AIR_CP"].copy(),
    ustar=df["USTAR"].copy(),
    swin=df["SW_IN_T1_47_1_gfXG"].copy(),
    lat=47.478333,  # CH–LAE
    lon=8.364389,  # CH–LAE
    utc_offset=1,
)
physics.run(correction_method_base="JAR09", gapfill=True)
results_physics_df = physics.get_results()

Aerodynamic resistance: Mean = 25.97 s m-1
Dry air density: Mean = 1.1448 kg m-3


Starting gap-filling for
FCT_UNSC
using <class 'sklearn.ensemble._forest.RandomForestRegressor'>

Adding new data columns ...
++ Added new columns with lagged variants for: ['TA_T1_47_1', 'AIR_CP', 'AIR_DENSITY', 'U', 'USTAR'] (lags between -1 and -1 with stepsize 1), no lagged variants for: ['FCT_UNSC']. Shifting the time series created gaps which were then filled with the nearest value.
++ Added new columns with timestamp info: ['.YEAR', '.SEASON', '.SEASON_SIN', '.SEASON_COS', '.MONTH', '.MONTH_SIN', '.MONTH_COS', '.WEEK', '.WEEK_SIN', '.WEEK_COS', '.DOY', '.DOY_SIN', '.DOY_COS', '.HOUR', '.HOUR_SIN', '.HOUR_COS', '.YEARMONTH', '.YEARDOY', '.YEARWEEK'] 
++ Added new column .RECORDNUMBER with record numbers from 1 to 181512.

Sanitizing timestamp ...
>>> Validating timestamp naming of timestamp column TIMESTAMP_MIDDLE ... Timestamp name OK.
>>> Converting timestamp TIMESTAMP_MIDDLE to datetime ... OK
>

In [9]:
results_physics_df.describe()

,FCT_UNSC_gfRF,FCT_UNSC,TS,TA_T1_47_1,AIR_CP,AERODYNAMIC_RESISTANCE,AIR_DENSITY,...,U,USTAR,CO2_MOLAR_DENSITY,SW_IN_T1_47_1_gfXG,AIR_THERMAL_CONDUCTIVITY,DAYTIME,LATENT_HEAT_VAPORIZATION_J_UMOL
count,249194.000000,181512.000000,260258.000000,260258.000000,224152.000000,203306.000000,224152.000000,...,224152.000000,224152.000000,207655.000000,262944.000000,260258.000000,262944.000000,260258.000000
mean,15.116406,15.558114,10.609026,8.505080,1010.501247,13.106494,1.151707,...,2.490855,0.488931,15968.675591,143.332497,0.024845,0.446217,0.044693
std,11.301625,12.383884,8.238440,8.210739,2.460679,11.128519,0.034942,...,1.805885,0.306027,2355.428950,236.465935,0.000575,0.497100,0.000351
min,0.872043,0.872043,-16.540001,-17.200001,975.647000,0.041636,1.053700,...,0.009557,0.007408,8483.380000,0.000000,0.023046,0.000000,0.043628
25%,7.228028,6.440754,4.248751,2.137056,1008.540000,5.547033,1.125710,...,1.160550,0.252767,14741.200000,0.000000,0.024400,0.000000,0.044435
50%,12.069695,12.107320,10.796000,8.560000,1010.090000,9.280123,1.149870,...,2.051600,0.432720,15549.500000,0.000000,0.024849,0.000000,0.044691
75%,19.652303,20.916502,16.737599,14.540000,1012.240000,16.775170,1.177020,...,3.362165,0.658582,16448.350000,196.028532,0.025268,1.000000,0.044965
max,103.919409,103.919409,34.840751,33.445910,1021.290000,85.902511,1.278340,...,13.249100,3.322560,37033.300000,1110.706662,0.026591,1.000000,0.045791


# Load scaling factors from `ScopOptimizer`

In [12]:
sfdf = pd.read_csv("32_SelfHeatingCorrection_ScalingFactors_NEE.csv")
sfdf

,TIMESTAMP_MIDDLE,NEE_L3.1_L3.2_QCF_IRGA72,CO2_MOLAR_DENSITY_IRGA72,AIR_CP_IRGA72,AIR_DENSITY_IRGA72,VAPOR_DENSITY_IRGA72,U_IRGA72,...,RH_T1_47_1_IRGA72,LE_L3.1_L3.2_QCF_IRGA72,H2O_MOLAR_DENSITY_IRGA72,NEE_L3.1_L3.2_QCF_IRGA75,CO2_MOLAR_DENSITY_IRGA75,LE_L3.1_L3.2_QCF_IRGA75,H2O_MOLAR_DENSITY_IRGA75
0,2016-05-27 00:15:00,NaN,14.8134,1013.51,1.10328,0.010099,2.380700,...,63.015150,16.447500,560.725,NaN,15.0989,NaN,560.417
1,2016-05-27 00:45:00,NaN,15.0169,1013.60,1.10696,0.010263,0.950929,...,67.925517,NaN,569.526,NaN,15.3100,NaN,569.512
2,2016-05-27 01:15:00,NaN,14.9983,1013.54,1.10644,0.010181,2.205350,...,66.753795,-5.834744,565.637,NaN,15.2706,-6.041330,565.009
3,2016-05-27 01:45:00,NaN,15.0355,1013.55,1.10705,0.010196,1.170180,...,67.506901,2.551110,566.157,NaN,15.3158,2.280256,565.825
4,2016-05-27 02:15:00,NaN,15.1012,1013.59,1.10911,0.010277,0.268074,...,70.223839,-12.065690,570.346,NaN,15.3908,-16.924510,570.299
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
27067,2017-12-11 21:45:00,NaN,16.4999,1009.22,1.14740,0.005345,5.951100,...,99.864659,NaN,294.161,NaN,19.2477,NaN,NaN
27068,2017-12-11 22:15:00,NaN,16.4604,1009.31,1.14648,0.005444,6.688060,...,99.864659,NaN,299.689,NaN,17.1550,NaN,302.108
27069,2017-12-11 22:45:00,NaN,16.4490,1009.33,1.14628,0.005483,5.815860,...,99.864659,NaN,301.782,NaN,16.2731,NaN,304.261
27070,2017-12-11 23:15:00,NaN,16.5297,1009.16,1.14953,0.005275,6.391870,...,99.864659,NaN,290.160,NaN,16.2659,NaN,292.735


# Correct open-path fluxes with `ScopApplicator`

In [14]:
applicator = ScopApplicator(
    flux_type="CO2",
    fct_unsc=results_physics_df["FCT_UNSC_gfRF"],
    ts=results_physics_df["TS"],
    ra=results_physics_df["AERODYNAMIC_RESISTANCE"],
    rho_d=results_physics_df["DRY_AIR_DENSITY"],
    scaling_factors_df=sfdf,
    flux_openpath=df["FC"].copy(),
    flux_closedpath=df["NEE_L3.1_L3.2_QCF_IRGA72"].copy(),
    classvar=df["USTAR"].copy(),
    daytime=results_physics_df["DAYTIME"].copy(),
    swin=df["SW_IN_T1_47_1_gfXG"].copy()
)
applicator.run()

KeyError: 'NEE_L3.1_L3.2_QCF_IRGA72'

In [ ]:
applicator.stats()

In [ ]:
applicator.plot_flux_analysis_dashboard();

In [ ]:
applicator.plot_diel_cycles()

# Scaling factors table

In [ ]:
scaling_factors_df

# Save to file

In [ ]:
filename = "32_SelfHeatingCorrection_ScalingFactors_NEE"
df.to_csv(f"{filename}.csv", index=True)

# ✅END OF NOTEBOOK

In [ ]:
dt_string = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
print(f"Finished. {dt_string}")